In [1]:
import napari
import nd2
import numpy as np
import scipy.ndimage as ndi
import cv2
import plotly.express as px
import pandas as pd
import glob
import skimage as ski
import dask

In [2]:
viewer = napari.Viewer()

In [43]:
channel = 1


In [44]:
def backsub(inp, radius=60):
    filterSize =(radius, radius)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                    filterSize)
    blurred = cv2.GaussianBlur(inp, (5, 5), 0)
    tophat_img = cv2.morphologyEx(blurred,
                                cv2.MORPH_TOPHAT,
                                kernel)
    rtn = inp.astype(np.single) - (blurred-tophat_img)
    rtn = np.clip(rtn, 0, np.inf)

    return rtn

def dask_sub(inp, radius=60):
    # Build a list of delayed plane tasks and compute them locally
    img = np.reshape(inp, (-1, inp.shape[-2], inp.shape[-1]))
    tasks = [dask.delayed(backsub)(i, radius) for i in img]
    results = dask.compute(*tasks)  # local compute via wrapper's thread pool
    rtn = np.reshape(np.array(results), inp.shape)
    return rtn

In [45]:
def get_peaks(img, viewer=None, display=False, threshold=0.1):
    LoG = ndi.gaussian_laplace(img.astype(np.float32), sigma=2)

    peaks = ski.feature.peak_local_max(-LoG, min_distance=1, threshold_abs=threshold, num_peaks=10000)

    backsub = dask_sub(img, radius=20)
    smoothed = ndi.gaussian_filter(backsub.astype(np.float32), sigma=2)
    ints = smoothed[peaks[:,0], peaks[:,1]]

    if display and viewer is not None:
        viewer.add_image(img)
        viewer.add_image(-LoG, colormap='magenta')
        viewer.add_points(peaks, size=8, name='peaks', face_color='green')

    return peaks, ints

def get_peaks_rel(img, viewer=None, display=False, threshold=0.1):
    LoG = ndi.gaussian_laplace(img.astype(np.float32), sigma=2)
    LoG = (LoG - np.percentile(LoG, 1)) / (np.percentile(LoG, 99.999) - np.percentile(LoG, 1))

    peaks = ski.feature.peak_local_max(-LoG, min_distance=1, threshold_abs=threshold, num_peaks=10000)

    backsub = dask_sub(img, radius=20)
    smoothed = ndi.gaussian_filter(backsub.astype(np.float32), sigma=2)
    ints = smoothed[peaks[:,0], peaks[:,1]]

    if display and viewer is not None:
        viewer.add_image(img)
        viewer.add_image(-LoG, colormap='magenta')
        viewer.add_points(peaks, size=8, name='peaks', face_color='green')

    return peaks, ints

In [46]:
fnames = glob.glob('*/*.tif')
fnames

['rep1_cen6g18r\\perturbed_rep1.tif',
 'rep1_cen6g18r\\perturbed_rep1_0001.tif',
 'rep1_cen6g18r\\perturbed_rep1_0002.tif',
 'rep1_cen6g18r\\unperturbed_rep1.tif',
 'rep1_cen6g18r\\unperturbed_rep1_0001.tif',
 'rep1_cen6g18r\\unperturbed_rep1_0002.tif',
 'rep2_cen6g18r\\perturbed_rep2.tif',
 'rep2_cen6g18r\\perturbed_rep2_0001.tif',
 'rep2_cen6g18r\\perturbed_rep2_0002.tif',
 'rep2_cen6g18r\\unperturbed_rep2.tif',
 'rep2_cen6g18r\\unperturbed_rep2_0001.tif',
 'rep2_cen6g18r\\unperturbed_rep2_0002.tif',
 'rep3_cen6g18r\\perturbed_rep3.tif',
 'rep3_cen6g18r\\perturbed_rep3_0001.tif',
 'rep3_cen6g18r\\perturbed_rep3_0002.tif',
 'rep3_cen6g18r\\unperturbed_rep3.tif',
 'rep3_cen6g18r\\unperturbed_rep3_0001.tif',
 'rep3_cen6g18r\\unperturbed_rep3_0002.tif']

In [47]:
fname = fnames[5]
img = ski.io.imread(fname)
cimg = img[1,:,:,channel]

In [48]:
# For C0 Used 1.0, for C1 Used 1.0
peak_threshold = 1.0

In [49]:
get_peaks_rel(cimg, viewer=viewer, display=True, threshold=peak_threshold)

(array([[1417, 1445],
        [1606, 2081],
        [1390, 1407],
        [2193, 1820],
        [ 960, 1527],
        [1618, 2085],
        [2227,  965],
        [1291,  131],
        [1961,  160],
        [1245,  131],
        [1814, 1514],
        [1020, 1521],
        [2217, 1806],
        [2266,  892],
        [1652,    5],
        [1782, 1509],
        [1776, 2228],
        [1636,   54],
        [1865,  277],
        [1868,  238],
        [2164,  758],
        [2146,  643],
        [1028,  470],
        [1043, 1314],
        [1042,  553],
        [1097,  536],
        [1141,  972],
        [1034,  445],
        [1093,  955],
        [1166, 1399],
        [ 636, 1670],
        [1088,  478],
        [ 828, 1528],
        [1160, 1498],
        [1045, 1306],
        [ 803,  812],
        [1935,  164],
        [1040, 2301],
        [ 314,  216],
        [1156, 1515],
        [ 136,  483],
        [1046,  536],
        [ 818,  783],
        [1064, 2081],
        [ 828, 1547],
        [ 

In [50]:
remake = True
if remake:
    lst = []
    for fname in fnames:
        img = ski.io.imread(fname)
        labels = ski.io.imread(fname.replace('.tif', '.tiff'))
        for idx, i in enumerate(img):
            cimg = i[:,:,channel]
            #peaks = get_peaks(cimg, threshold=200.0)
            #peaks, ints = get_peaks_rel(cimg, threshold=1.0)
            peaks, ints = get_peaks_rel(cimg, threshold=peak_threshold)
            df = pd.DataFrame(peaks, columns=['y', 'x'])
            df['z'] = idx
            df['label'] = labels[idx, peaks[:,0], peaks[:,1]]
            df['intensity'] = ints
            df['fname'] = fname
            lst.append(df)
    df = pd.concat(lst)
    df.to_csv('Raw_C' + str(channel) + '_PeakData.csv', index=False) 

# Analyze

In [51]:
df = pd.read_csv('Raw_C' + str(channel) + '_PeakData.csv')

In [52]:
df['rep'] = df['fname'].str.split('_').str[0]

In [53]:
df['condition'] = 'perturbed'
df.loc[df['fname'].str.contains('unperturbed'), 'condition'] = 'unperturbed'

In [54]:
agged = df.groupby(['fname', 'z', 'label', 'rep', 'condition']).agg({'x':len, 'intensity':[np.max, np.median, np.min]}).reset_index().rename(columns={'x':'num_peaks'})
agged.columns = ['fname', 'z', 'label', 'rep', 'condition', 'num_peaks', 'intensity_max', 'intensity_median', 'intensity_min']
agged = agged[agged['label'] > 0]
agged

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\476421915.py:1: FutureWarning:

The provided callable <function max at 0x000002B4C2569120> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\476421915.py:1: FutureWarning:

The provided callable <function median at 0x000002B4C26DCCC0> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\476421915.py:1: FutureWarning:

The provided callable <function min at 0x000002B4C2569260> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.



,fname,z,label,rep,condition,num_peaks,intensity_max,intensity_median,intensity_min
1,rep1_cen6g18r\perturbed_rep1.tif,0,3,rep1,perturbed,2,30899.951,29352.5935,27805.236
2,rep1_cen6g18r\perturbed_rep1.tif,0,4,rep1,perturbed,2,21041.967,19817.3555,18592.744
3,rep1_cen6g18r\perturbed_rep1.tif,0,5,rep1,perturbed,2,20317.527,16376.5535,12435.580
4,rep1_cen6g18r\perturbed_rep1.tif,0,6,rep1,perturbed,2,22855.766,18701.6495,14547.533
5,rep1_cen6g18r\perturbed_rep1.tif,0,8,rep1,perturbed,1,12754.919,12754.9190,12754.919
...,...,...,...,...,...,...,...,...,...
65910,rep3_cen6g18r\unperturbed_rep3_0002.tif,47,63,rep3,unperturbed,2,35642.652,34760.2660,33877.880
65911,rep3_cen6g18r\unperturbed_rep3_0002.tif,47,66,rep3,unperturbed,2,37798.836,37061.0830,36323.330
65912,rep3_cen6g18r\unperturbed_rep3_0002.tif,47,67,rep3,unperturbed,2,27018.953,26531.3630,26043.773
65913,rep3_cen6g18r\unperturbed_rep3_0002.tif,47,68,rep3,unperturbed,2,42614.996,41268.9100,39922.824


# Filter

In [55]:
tagged = agged.copy()
tagged = tagged.groupby(['fname', 'z', 'condition', 'rep']).agg({'z':len, 'num_peaks':[lambda x: np.sum(x==0), lambda x: np.sum(x==1), lambda x: np.sum(x==2), lambda x: np.sum(x==3), lambda x: np.sum(x==4), lambda x: np.sum(x>4)]}).rename(columns={'z':'num_labels'}).reset_index()
tagged.columns = ['fname', 'z', 'condition', 'rep', 'num_labels', 'num_0_peaks', 'num_1_peaks', 'num_2_peaks', 'num_3_peaks', 'num_4_peaks', 'num_more_than_4_peaks']

tagged['fraction_0_peaks'] = tagged['num_0_peaks'] / tagged['num_labels']
tagged['fraction_1_peaks'] = tagged['num_1_peaks'] / tagged['num_labels']
tagged['fraction_2_peaks'] = tagged['num_2_peaks'] / tagged['num_labels']
tagged['fraction_3_peaks'] = tagged['num_3_peaks'] / tagged['num_labels']
tagged['fraction_4_peaks'] = tagged['num_4_peaks'] / tagged['num_labels']
tagged['fraction_more_peaks'] = tagged['num_more_than_4_peaks'] / tagged['num_labels']
tagged['file'] = tagged['fname']
f = px.violin(tagged, x='fname', y='fraction_2_peaks', points='all', range_y=[0,1], width=1200, hover_data=['z'], height=500)
f

In [56]:
from os import path

def get_bad_list():
    if path.exists("Bad_List.csv"):
        bad_list = pd.read_csv('Bad_List.csv').drop(['Unnamed: 0'], axis=1)
    else:
        bad_list = pd.DataFrame(columns=['fname', 'z'])
    return bad_list

In [57]:
import plotly.graph_objects as go

bad_list = get_bad_list()
merge_list = bad_list.copy()
merge_list['Good'] = False
filtered_tagged = tagged.merge(merge_list, on=['fname', 'z'], how='left').fillna(True)
filtered_tagged = filtered_tagged[filtered_tagged['Good']]

#filtered_df = df[~((df['fname'].isin(bad_list['File']) & ())]

f=go.FigureWidget(
    px.violin(filtered_tagged, x='file', y='fraction_2_peaks', points='all', range_y=[-0.05,1.05], width=1200, hover_data=['fname', 'z'], height=500)
    )
def selection_fn(trace,points,selector):
    
    if (len(points.point_inds)>0):
        
        global bad_list
        
        file = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        z = f.data[points.trace_index]['customdata'][points.point_inds[-1]][1]

        # #bad_list = bad_list.append(pd.DataFrame([[file]], columns=bad_list.columns))
        bad_list = pd.concat([bad_list, pd.DataFrame([[file,z]], columns=bad_list.columns)])
        bad_list.to_csv('Bad_List.csv')
        print(file)
     
                
def click_fn(trace, points, state):
    
    if (len(points.point_inds)>0):
        print(f.data[points.trace_index]['customdata'][points.point_inds[-1]])
        fname = f.data[points.trace_index]['customdata'][points.point_inds[-1]][0]
        z = f.data[points.trace_index]['customdata'][points.point_inds[-1]][1]
        
        img = ski.io.imread(fname)
        i = img[int(z)]
        cimg = i[:,:,channel]
        viewer.layers.clear()
        #get_peaks(cimg, threshold=200.0, viewer=viewer, display=True)
        get_peaks_rel(cimg, viewer=viewer, display=True, threshold=peak_threshold)

        labels = ski.io.imread(fname.replace('.tif', '.tiff'))
        L = labels[int(z)]
        viewer.add_labels(L)
        viewer.layers[-1].contour = 1
        viewer.layers[0].name = fname + '__' + str(z)

        viewer.add_image(i[:,:,-1], name='DAPI')
        
        

f.data[0].on_selection(selection_fn)

f.data[0].on_click(click_fn)

f

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\1034721436.py:6: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'box': {'visible': False},
              'customdata': array([['rep1_cen6g18r\\perturbed_rep1.tif', 0],
                                   ['rep1_cen6g18r\\perturbed_rep1.tif', 1],
                                   ['rep1_cen6g18r\\perturbed_rep1.tif', 2],
                                   ...,
                                   ['rep3_cen6g18r\\unperturbed_rep3_0002.tif', 45],
                                   ['rep3_cen6g18r\\unperturbed_rep3_0002.tif', 46],
                                   ['rep3_cen6g18r\\unperturbed_rep3_0002.tif', 47]],
                                  shape=(1182, 2), dtype=object),
              'hovertemplate': ('file=%{x}<br>fraction_2_peaks=' ... '{customdata[1]}<extra></extra>'),
              'legendgroup': '',
              'marker': {'color': '#636efa'},
              'name': '',
              'offsetgroup': '',
              'orientation': 'v',
              'points': 'all',
    

In [58]:
bad_list = get_bad_list()
merge_list = bad_list.copy()
merge_list['Good'] = False

filtered_tagged = tagged.merge(merge_list, on=['fname', 'z'], how='left').fillna(True)
filtered_tagged = filtered_tagged[filtered_tagged['Good']]

agged = agged.merge(merge_list, on=['fname', 'z'], how='left').fillna(True)
agged = agged[agged['Good']]

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\1858917021.py:5: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\1858917021.py:8: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [59]:
f = px.violin(filtered_tagged, x='condition', y='fraction_2_peaks', points='all', range_y=[0,1], width=1200, hover_data=['fname', 'z'], height=500, color='rep')
f.write_html('C' + str(channel) + '_Perturbed_vs_Unperturbed.html')
f

In [60]:
filtered_tagged.to_csv('C'+str(channel)+'_Results.csv')

# Looking at triples

In [61]:
trip_agged = agged[agged['num_peaks'] == 3]

In [62]:
trip_agged['p2'] = trip_agged['intensity_max'] / trip_agged['intensity_max']
trip_agged['p1'] = trip_agged['intensity_median'] / trip_agged['intensity_max']
trip_agged['p0'] = trip_agged['intensity_min'] / trip_agged['intensity_max']

trip_agged['p1_normed'] = (trip_agged['intensity_median'] - trip_agged['intensity_min']) / (trip_agged['intensity_max'] - trip_agged['intensity_min'])

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\3711035823.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\3711035823.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\smc\AppData\Local\Temp\ipykernel_32260\3711035823.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs

In [63]:
f = px.violin(trip_agged, x='condition', y='p1_normed', points='all', width=900, hover_data=['fname', 'z'], height=500, color='rep')
f.write_html('C' + str(channel) + '_Triplet_p1normed_Perturbed_vs_Unperturbed.html')
f

In [64]:
trip_agged.to_csv('C'+str(channel)+'_Triplet_Results.csv')